# Frozen ImageNet baseline through the Phikon pipeline (about 45–60 min on a T4 GPU)
**Purpose.** In the paper, Phikon (frozen, pathology-pretrained) is compared with fine-tuned ImageNet-pretrained CNNs. This notebook
runs **frozen ImageNet-pretrained backbones through exactly the same pipeline as Phikon** (224 × 224 input, frozen embedding,
standardisation, logistic regression with C chosen on the validation set only, the same 12-tile LungHist700 scheme, same partitions,
seeds 42–46, both protocols). This separates the effect of the *pretraining domain* (pathology versus ImageNet) from the effect of the
*adaptation regime* (frozen linear probe versus fine-tuning).

Backbones: **ViT-B/16 ImageNet** (`google/vit-base-patch16-224`, same architecture as Phikon) and **EfficientNetB0 ImageNet**
(Keras weights, the same backbone as the fine-tuned models in the paper).

**How to run (Google Colab):** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Allow Google Drive access.
Results are saved to `MyDrive/imagenet_frozen_results`. If Colab disconnects, run all cells again – finished parts are skipped.
At the end a SUMMARY is printed and `imagenet_frozen_results.zip` is downloaded. **Send both to Claude.**

If the earlier Phikon results are still in `MyDrive/extra_results/`, the notebook also (i) checks that the partitions are identical to
those used in the paper and (ii) computes paired Phikon-minus-ImageNet differences with patient-bootstrap CIs. Nothing else is needed.

In [ ]:
# %% Settings
import os
TEST_MODE = os.environ.get("GS_TEST_MODE") == "1"     # offline test only; leave as is
SEEDS      = [42, 43, 44, 45, 46]
TILE       = 400
N_BOOT     = 2000
FM_BATCH   = 64
BACKBONES  = {"ViT-B16_ImageNet": "google/vit-base-patch16-224",      # ImageNet-21k pretrained, ImageNet-1k fine-tuned, 224 px
              "EfficientNetB0_ImageNet": "keras:EfficientNetB0"}       # Keras ImageNet weights (backbone of the paper's CNNs)
GROUPS_COMMIT = "912f342fe2a781e847f395d79982dad4d3d425ae"
GROUPS_SHA256 = "2ff9e992f759f265dfc9aa2363de0dfcc8e6559884d8bb02a4c925214fdf7e23"
GROUPS_URL = f"https://raw.githubusercontent.com/GeorgeBatch/LC25000-clean/{GROUPS_COMMIT}/kaggle/lc25000_image_groups.csv"
LH_URL     = "https://ndownloader.figshare.com/files/45206104"
SAVE_DIR   = "/content/drive/MyDrive/imagenet_frozen_results"
REF_DIR    = "/content/drive/MyDrive/extra_results"            # earlier Phikon run (splits + external probabilities), optional
DATA_DIR   = None
LH_ROOT    = "/content/lunghist700"
if TEST_MODE:
    N_BOOT, SEEDS, FM_BATCH = 100, [42, 43], 8
    DATA_DIR, LH_ROOT, SAVE_DIR = os.environ["GS_DATA"], os.environ["GS_LH"], os.environ["GS_OUT"]
    REF_DIR = os.environ.get("GS_REF", "/nonexistent")
    GROUPS_URL = os.environ["GS_GROUPS"]

In [ ]:
# %% Imports and shared helpers
import json, time, hashlib, random, glob, re, subprocess, zipfile, platform
import numpy as np, pandas as pd, cv2
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
CLASSES = ["lung_aca", "lung_n", "lung_scc"]
if not TEST_MODE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)

def ece_score(p, y, n_bins=15):
    conf, pred = p.max(1), p.argmax(1); bins, e = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def cluster_ci(correct, clusters, seed=0):
    rng = np.random.default_rng(seed); u = np.unique(clusters); idx = {k: np.where(clusters == k)[0] for k in u}; a = []
    for _ in range(N_BOOT):
        s = np.concatenate([idx[k] for k in rng.choice(u, len(u), replace=True)]); a.append(correct[s].mean())
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def metrics(p, y, clusters):
    yp = p.argmax(1); cm = confusion_matrix(y, yp, labels=[0, 1, 2]); rec = cm.diagonal() / cm.sum(1).clip(min=1)
    return {"n": int(len(y)), "accuracy": float((yp == y).mean()),
            "accuracy_95ci_cluster": cluster_ci((yp == y).astype(float), clusters),
            "macro_f1": float(f1_score(y, yp, average="macro")), "balanced_accuracy": float(balanced_accuracy_score(y, yp)),
            "ece": ece_score(p, y), "brier": float(((p - np.eye(3)[y]) ** 2).sum(1).mean()),
            "recall": dict(zip(CLASSES, rec.round(4).tolist())),
            "auc_ovr_macro": float(roc_auc_score(y, p, multi_class="ovr")) if len(np.unique(y)) == 3 else None,
            "errors": int((yp != y).sum()), "confusion_matrix": cm.tolist()}

def agg(vals):
    v = np.array(vals, float); m = v.mean(); sd = v.std(ddof=1) if len(v) > 1 else 0.0
    return {"mean": float(m), "sd": float(sd), "min": float(v.min()), "max": float(v.max()), "per_seed": v.round(5).tolist()}

In [ ]:
# %% LC25000 (MD5-deduplicated) + source-tile groups + both partitioning protocols (identical to the earlier notebooks)
def find_lung_dir(root):
    for d, ds, _ in os.walk(root):
        if all(c in ds for c in CLASSES): return d
    raise FileNotFoundError(root)
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = find_lung_dir(kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images"))
else:
    DATA_DIR = find_lung_dir(DATA_DIR)
rows = []
for label, c in enumerate(CLASSES):
    for f in sorted(glob.glob(os.path.join(DATA_DIR, c, "*"))):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            with open(f, "rb") as fh:
                rows.append({"path": f, "label": label, "cls": c, "filename": os.path.basename(f), "md5": hashlib.md5(fh.read()).hexdigest()})
df_all = pd.DataFrame(rows)
df = df_all.drop_duplicates("md5", keep="first").reset_index(drop=True)
conflict = df_all.groupby("md5")["label"].nunique()
df = df[~df["md5"].isin(conflict[conflict > 1].index)].reset_index(drop=True)
groups = pd.read_csv(GROUPS_URL)
if not TEST_MODE:
    import urllib.request
    assert hashlib.sha256(urllib.request.urlopen(GROUPS_URL).read()).hexdigest() == GROUPS_SHA256, "grouping file changed - send this message to Claude"
groups = groups[groups["label"].isin(CLASSES)][["filename", "label", "group_id"]].rename(columns={"label": "g_label"})
df = df.merge(groups, on="filename", how="left")
assert df["group_id"].notna().all(), "images without group - send this message to Claude"
assert (df["g_label"] == df["cls"]).all()
df["group_id"] = df["group_id"].astype(int)
print(f"LC25000 lung images kept: {len(df)} | groups: {df['group_id'].nunique()}")

def image_split(df, seed):
    """Protocol A: conventional stratified image-level 70/15/15 split (seed 42 reproduces the first benchmark)."""
    tr, tmp = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=seed)
    va, te = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=seed)
    parts = {"train": tr.reset_index(drop=True), "val": va.reset_index(drop=True), "test": te.reset_index(drop=True)}
    for a in parts:
        for b in parts:
            if a < b: assert not set(parts[a]["md5"]) & set(parts[b]["md5"]), "hash overlap!"
    return parts

def group_split(df, seed, fr=(0.70, 0.15, 0.15)):
    """Protocol B: identical to the group-split notebook (whole source groups, largest relative shortfall)."""
    rng = np.random.default_rng(seed); assign = {}; keys = ["train", "val", "test"]
    for c in CLASSES:
        sizes = df[df["cls"] == c].groupby("group_id").size()
        g = sizes.index.values.copy(); rng.shuffle(g)
        target = np.array(fr) * sizes.sum(); filled = np.zeros(3)
        for gid in g:
            k = int(np.argmax((target - filled) / target)); assign[gid] = keys[k]; filled[k] += sizes[gid]
    s = df.assign(split=df["group_id"].map(assign))
    parts = {k: s[s["split"] == k].reset_index(drop=True) for k in keys}
    for a in parts:
        for b in parts:
            if a < b:
                assert not set(parts[a]["group_id"]) & set(parts[b]["group_id"]), "group overlap!"
                assert not set(parts[a]["md5"]) & set(parts[b]["md5"]), "hash overlap!"
    return parts

SPLITS = {("A", s): image_split(df, s) for s in SEEDS}
SPLITS.update({("B", s): group_split(df, s) for s in SEEDS})
leak = {s: float(100 * SPLITS[("A", s)]["test"]["group_id"].isin(set(SPLITS[("A", s)]["train"]["group_id"])).mean()) for s in SEEDS}
print("Protocol A leakage (% of test images whose source group is in training):", leak)
for (p, s), parts in SPLITS.items():
    pass

# Check that the partitions are identical to those of the paper (if the earlier run is on Drive)
n_checked = 0
for (p, s), parts in SPLITS.items():
    d = os.path.join(REF_DIR, f"splits/protocol{p}_seed{s}")
    for k in ["train", "val", "test"]:
        f = os.path.join(d, f"{k}.csv")
        if os.path.exists(f):
            ref = pd.read_csv(f)
            assert sorted(ref["filename"]) == sorted(parts[k]["filename"]), f"partition differs: protocol {p} seed {s} {k} - send this message to Claude"
            n_checked += 1
print(f"Partitions checked against the earlier run: {n_checked} of {3 * len(SPLITS)} files identical"
      + ("" if n_checked else " (earlier splits not found on Drive - partitions are regenerated with the same code and seeds)"))

In [ ]:
# %% LungHist700 index (external only; no tuning)
if not glob.glob(LH_ROOT + "/**/*.jp*g", recursive=True):
    os.makedirs(LH_ROOT, exist_ok=True); rar = "/content/LungHist700.rar"
    if not os.path.exists(rar): subprocess.run(["wget", "-q", "-O", rar, LH_URL], check=True)
    if subprocess.run(["which", "unrar"], capture_output=True).returncode != 0:
        subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True)
    if subprocess.run(["unrar", "x", "-o+", "-inul", rar, LH_ROOT + "/"]).returncode != 0:
        subprocess.run(["7z", "x", "-y", f"-o{LH_ROOT}", rar], check=True, stdout=subprocess.DEVNULL)
rows = []
for f in sorted(glob.glob(LH_ROOT + "/**/*.jp*g", recursive=True)):
    m = re.match(r"^(aca|nor|scc)(?:_(bd|md|pd))?_(20x|40x)_(\d+)\.jpe?g$", os.path.basename(f).lower())
    if m:
        sup, sub, res_, iid = m.groups()
        rows.append({"path": f, "superclass": sup, "subclass": sub or "", "resolution": res_, "image_id": int(iid),
                     "label": {"aca": 0, "nor": 1, "scc": 2}[sup]})
ext = pd.DataFrame(rows)
for c in glob.glob(LH_ROOT + "/**/*.csv", recursive=True):
    meta = pd.read_csv(c)
    if {"superclass", "resolution", "image_id", "patient_id"} <= set(meta.columns):
        meta = meta.copy()
        if "subclass" not in meta: meta["subclass"] = ""
        meta["subclass"] = meta["subclass"].fillna("").astype(str).str.lower().replace("nan", "")
        meta["superclass"] = meta["superclass"].astype(str).str.lower(); meta["resolution"] = meta["resolution"].astype(str).str.lower()
        meta["image_id"] = meta["image_id"].astype(int)
        ext = ext.merge(meta[["superclass", "subclass", "resolution", "image_id", "patient_id"]],
                        on=["superclass", "subclass", "resolution", "image_id"], how="left"); break
assert "patient_id" in ext and ext["patient_id"].notna().all(), "patient IDs missing - send this message to Claude"
y_ext, pat = ext["label"].values, ext["patient_id"].values
T = 12
print(f"LungHist700: {len(ext)} images, {ext['patient_id'].nunique()} patients")

def ext_tiles(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    out = [img[yy:yy + TILE, xx:xx + TILE] for yy in range(0, H - TILE + 1, TILE) for xx in range(0, W - TILE + 1, TILE)]
    assert len(out) == T, f"{path}: {len(out)} tiles instead of 12 - send this message to Claude"
    return out

def ext_result(p):
    res = metrics(p, y_ext, pat)
    res["by_resolution"] = {r: float((p.argmax(1)[ext["resolution"] == r] == y_ext[ext["resolution"] == r]).mean()) for r in ["20x", "40x"]}
    pp = ext[["patient_id", "label"]].assign(**{f"p{i}": p[:, i] for i in range(3)}).groupby(["patient_id", "label"], as_index=False).mean()
    res["patient_class_accuracy"] = float((pp[["p0", "p1", "p2"]].to_numpy().argmax(1) == pp["label"].to_numpy()).mean())
    res["patient_class_units"] = int(len(pp))
    return res

In [ ]:
# %% Frozen ImageNet features (extracted once per backbone and cached on Drive)
import torch
from PIL import Image
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")
FEAT = {}
for name, src in BACKBONES.items():
    fdir = os.path.join(SAVE_DIR, name); os.makedirs(fdir, exist_ok=True)
    f_int, f_ext = os.path.join(fdir, "features_lc25000.npy"), os.path.join(fdir, "features_lunghist700_tiles.npy")
    if not (os.path.exists(f_int) and os.path.exists(f_ext)):
        t0 = time.time(); info = {"backbone": name, "source": src}
        if src.startswith("keras:"):
            import tensorflow as tf, keras
            for g in tf.config.list_physical_devices("GPU"):
                try: tf.config.experimental.set_memory_growth(g, True)
                except RuntimeError: pass
            net = keras.applications.EfficientNetB0(include_top=False, weights=None if TEST_MODE else "imagenet",
                                                    input_shape=(224, 224, 3), pooling="avg")
            def embed(images):        # images: list of RGB uint8 arrays; Keras EfficientNet rescales internally
                x = np.stack([cv2.resize(im, (224, 224), interpolation=cv2.INTER_LINEAR) for im in images]).astype(np.float32)
                return net(x, training=False).numpy()
            info.update({"embedding": "global average pooling of the last convolutional block", "input": "224x224 bilinear, Keras internal scaling",
                         "tensorflow": tf.__version__, "keras": keras.__version__, "precision": "float32"})
        else:
            from transformers import AutoImageProcessor, AutoModel
            if TEST_MODE:
                from transformers import ViTConfig, ViTModel, ViTImageProcessor
                fm = ViTModel(ViTConfig(hidden_size=32, num_hidden_layers=1, num_attention_heads=2, intermediate_size=64, image_size=224, patch_size=32))
                proc = ViTImageProcessor(size={"height": 224, "width": 224}); rev = "test"
            else:
                from huggingface_hub import HfApi
                try: rev = HfApi().model_info(src).sha
                except Exception as e: print("Could not read the model revision (", e, ") - using main"); rev = "main"
                proc = AutoImageProcessor.from_pretrained(src, revision=rev); fm = AutoModel.from_pretrained(src, revision=rev)
            fm = fm.to(dev).eval()
            @torch.no_grad()
            def embed(images):
                x = proc(images=[Image.fromarray(im) for im in images], return_tensors="pt")["pixel_values"].to(dev)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(dev == "cuda")):
                    out = fm(pixel_values=x).last_hidden_state[:, 0, :]       # CLS token, as for Phikon
                return out.float().cpu().numpy()
            import transformers
            info.update({"revision": rev, "embedding": "CLS token of last hidden state (as for Phikon)", "processor": proc.to_dict(),
                         "torch": torch.__version__, "transformers": transformers.__version__,
                         "precision": "float16 autocast on GPU" if dev == "cuda" else "float32 (CPU)"})
        feats, buf = [], []
        for i, f in enumerate(df["path"]):
            buf.append(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB))
            if len(buf) == FM_BATCH or i == len(df) - 1:
                feats.append(embed(buf)); buf = []
            if i % 2000 == 0: print(f"  {name}: LC25000 features {i}/{len(df)}  ({time.time() - t0:.0f} s)")
        np.save(f_int, np.concatenate(feats).astype(np.float32))
        feats, buf = [], []
        for i, f in enumerate(ext["path"]):
            buf += ext_tiles(f)
            if len(buf) >= FM_BATCH * 2 or i == len(ext) - 1:
                feats.append(embed(buf)); buf = []
        np.save(f_ext, np.concatenate(feats).astype(np.float32))
        info.update({"dim": int(feats[0].shape[1]), "n_lc25000_images": int(len(df)), "n_lunghist700_tiles": int(len(ext) * T),
                     "minutes": round((time.time() - t0) / 60, 1)})
        json.dump(info, open(os.path.join(fdir, "feature_info.json"), "w"), indent=1)
        if src.startswith("keras:"): del net
        else: del fm
        if dev == "cuda": torch.cuda.empty_cache()
    FEAT[name] = (np.load(f_int), np.load(f_ext))
    assert len(FEAT[name][0]) == len(df) and len(FEAT[name][1]) == len(ext) * T, f"{name}: feature count mismatch - delete its folder on Drive and rerun"
    print(f"{name}: features {FEAT[name][0].shape}, {FEAT[name][1].shape}")

In [ ]:
# %% Linear probe: identical to the Phikon baseline (standardise, choose C on validation only, test once)
row_of = {p: i for i, p in enumerate(df["path"])}
C_GRID = [0.001, 0.01, 0.1, 1.0]

def fit_probe(X, Xt, parts):
    itr, iva, ite = [[row_of[p] for p in parts[k]["path"]] for k in ["train", "val", "test"]]
    sc = StandardScaler().fit(X[itr]); best = None
    for C in C_GRID:
        clf = LogisticRegression(C=C, max_iter=3000).fit(sc.transform(X[itr]), df["label"].values[itr])
        acc = (clf.predict(sc.transform(X[iva])) == df["label"].values[iva]).mean()
        if best is None or acc > best[0]: best = (acc, C, clf)
    _, C, clf = best
    return C, clf.predict_proba(sc.transform(X[ite])), clf.predict_proba(sc.transform(Xt)).reshape(len(ext), T, 3).mean(1)

for name, (X, Xt) in FEAT.items():
    for (prot, s), parts in SPLITS.items():
        out = os.path.join(SAVE_DIR, name, f"protocol{prot}_seed{s}.json")
        if os.path.exists(out): continue
        C, p_test, p_ext = fit_probe(X, Xt, parts)
        res = {"backbone": name, "protocol": prot, "seed": s, "C": C, "C_at_grid_boundary": C in (C_GRID[0], C_GRID[-1]),
               "internal": metrics(p_test, parts["test"]["label"].values, parts["test"]["group_id"].values), "external": ext_result(p_ext)}
        json.dump(res, open(out, "w"), indent=1)
        np.save(os.path.join(SAVE_DIR, name, f"protocol{prot}_seed{s}_external_probs.npy"), p_ext)
        print(f"{name} | Protocol {prot} seed {s}: C={C} internal {100*res['internal']['accuracy']:.2f}% | "
              f"external {100*res['external']['accuracy']:.1f}% (balanced {100*res['external']['balanced_accuracy']:.1f}%)")

In [ ]:
# %% SUMMARY (copy everything printed below and send it to Claude together with imagenet_frozen_results.zip)
def paired_boot(pa, pb, seed=0):
    """Patient-cluster bootstrap of the external accuracy difference (a minus b), percentage points."""
    ca, cb = (pa.argmax(1) == y_ext).astype(float), (pb.argmax(1) == y_ext).astype(float)
    rng = np.random.default_rng(seed); u = np.unique(pat); idx = {k: np.where(pat == k)[0] for k in u}; d = []
    for _ in range(N_BOOT):
        s_ = np.concatenate([idx[k] for k in rng.choice(u, len(u), replace=True)]); d.append(ca[s_].mean() - cb[s_].mean())
    return float(100 * (ca.mean() - cb.mean())), [float(100 * np.percentile(d, 2.5)), float(100 * np.percentile(d, 97.5))]

S = {"backbones": {}, "phikon_minus_imagenet": {}}
for name in BACKBONES:
    S["backbones"][name] = {}
    for prot in ["A", "B"]:
        R = [json.load(open(os.path.join(SAVE_DIR, name, f"protocol{prot}_seed{s}.json"))) for s in SEEDS]
        m = {f"{d}_{k}": agg([r[d][k] for r in R]) for d in ["internal", "external"] for k in ["accuracy", "macro_f1", "balanced_accuracy", "ece", "brier", "auc_ovr_macro"]}
        m["internal_errors"] = [r["internal"]["errors"] for r in R]
        m["external_patient_class_accuracy"] = agg([r["external"]["patient_class_accuracy"] for r in R])
        m["external_recall"] = {c: agg([r["external"]["recall"][c] for r in R]) for c in CLASSES}
        m["external_by_resolution"] = {z: agg([r["external"]["by_resolution"][z] for r in R]) for z in ["20x", "40x"]}
        m["C"] = [r["C"] for r in R]
        S["backbones"][name][prot] = m
        fp = [os.path.join(REF_DIR, "foundation_model", f"protocol{prot}_seed{s}_external_probs.npy") for s in SEEDS]
        if all(os.path.exists(f) for f in fp):
            S["phikon_minus_imagenet"][f"{name}_{prot}"] = [
                paired_boot(np.load(f), np.load(os.path.join(SAVE_DIR, name, f"protocol{prot}_seed{s}_external_probs.npy")))
                for f, s in zip(fp, SEEDS)]
json.dump(S, open(os.path.join(SAVE_DIR, "imagenet_frozen_summary.json"), "w"), indent=1)

def f(a, sc=100, d=2): return f"{sc*a['mean']:.{d}f} ± {sc*a['sd']:.{d}f}"
print("=" * 78 + "\nSUMMARY - frozen ImageNet baselines through the Phikon pipeline, seeds", SEEDS)
for name in BACKBONES:
    info = json.load(open(os.path.join(SAVE_DIR, name, "feature_info.json")))
    print(f"\n{name}: " + json.dumps({k: v for k, v in info.items() if k != "processor"}))
    for prot in ["A", "B"]:
        m = S["backbones"][name][prot]
        print(f"  Protocol {prot}: internal {f(m['internal_accuracy'])}% (errors {m['internal_errors']}) | external {f(m['external_accuracy'],100,1)}% | "
              f"balanced {f(m['external_balanced_accuracy'],100,1)}% | macro-F1 {f(m['external_macro_f1'],1,3)} | AUC {f(m['external_auc_ovr_macro'],1,3)} | "
              f"ECE {f(m['external_ece'],1,3)}")
        print(f"     external recall aca/nor/scc {f(m['external_recall']['lung_aca'],1,2)} / {f(m['external_recall']['lung_n'],1,2)} / "
              f"{f(m['external_recall']['lung_scc'],1,2)} | 20x {f(m['external_by_resolution']['20x'],100,1)}% | 40x {f(m['external_by_resolution']['40x'],100,1)}% | "
              f"patient-class {f(m['external_patient_class_accuracy'],100,1)}% | C {m['C']}")
        k = f"{name}_{prot}"
        if k in S["phikon_minus_imagenet"]:
            print("     Phikon minus this baseline (external, pp, patient-bootstrap 95% CI) per seed: "
                  + "; ".join(f"{d:+.1f} [{lo:+.1f}, {hi:+.1f}]" for d, (lo, hi) in S["phikon_minus_imagenet"][k]))
print("=" * 78)
if not TEST_MODE:
    subprocess.run(f"cd {os.path.dirname(SAVE_DIR)} && zip -qr /content/imagenet_frozen_results.zip {os.path.basename(SAVE_DIR)} -x '*.npy'", shell=True)
    subprocess.run(f"cd {os.path.dirname(SAVE_DIR)} && zip -qr /content/imagenet_frozen_results.zip {os.path.basename(SAVE_DIR)} -i '*external_probs.npy'", shell=True)
    from google.colab import files; files.download("/content/imagenet_frozen_results.zip")